In [3]:
import os
import torch
import matplotlib.pyplot as plt


In [4]:
# Assign directories

# AF3 directories
AF_INPUT_PATH = '../af3/af_input/'
AF_OUTPUT_PATH = '../af3/af_output/'

# Structures directory
STRUCTURES_PATH = '../data/structures/'

# Data directories
DATA_PATH = '../data/'
RESIDUES_DIR = 'lists_of_residues/'
CHAIN_LENGTHS_DIR = 'chain_lengths/'
DCCM_MAPS_DIR = 'dccm_maps/'
TORCH_DIR = 'torch/'

# Training directories
CHECKPOINTS_PATH = '../checkpoints/'

# Original ANTIPASTI data directory
OLD_DATA_PATH = '../OLD/data/'

# Model inference

## Inference with original ANTIPASTI

In [247]:
import numpy as np
from antipasti.utils.torch_utils import load_checkpoint

In [ ]:
data_filename = f"AbCov-SARSCoV2-RBD_finetune-all"
data_filepath = os.path.join(DATA_PATH, TORCH_DIR, data_filename + '.pt')

test_x, test_y = torch.load(data_filepath)


In [ ]:
modes = 'all'
n_filters = 4
filter_size = 4
pooling_size = 1
n_max_epochs = 1044

input_shape = 292

path = '../checkpoints/full_ags_all_modes/model_epochs_' + str(n_max_epochs) + '_modes_' + str(modes) + '_pool_' + str(pooling_size) + '_filters_' + str(n_filters) + '_size_' + str(filter_size) + '.pt'
model,optimiser,_,train_losses, test_losses = load_checkpoint(path, input_shape)
model.eval()


## Modify

In [190]:
# Modify as appropriate

data_name = 'AbCov-SARSCoV2-RBD_with-ag'
# data_name = 'AbCov-SARSCoV2-RBD_original'
# data_name = 'AbCov-SARSCoV2-RBD_with-offblock'
# data_name = 'AbCov-SARSCoV2-RBD_finetune'
seed = 4
test_size = 0.1
confident = 30.7

data_filename = f"{data_name}-training-data_seed_{seed}_split_{test_size}"
# data_filename = data_name + '-training-data_seed_' + str(seed) + '_confident_' + str(confident) + '_split_' + str(test_size)

data_filepath = os.path.join(DATA_PATH, TORCH_DIR, data_filename + '.pt')

# epochs = 500
# epochs = 300
epochs = 100
modes = 'all'
pooling_size = 1
n_filters = 4
filter_size = 4
learning_rate = 5e-05
# learning_rate = 0.0005

weighted_loss = ""
# downweighting = 2
# weighted_loss = f"_downweighting_{downweighting}"

# model_name = 'finetunedmodel_epochs_' + str(epochs) + '_modes_' + str(modes) + '_pool_' + str(pooling_size) + '_filters_' + str(n_filters) + '_size_' + str(filter_size) + '.pt'
# model_name = 'extendedmodel_epochs_' + str(epochs) + '_modes_' + str(modes) + '_pool_' + str(pooling_size) + '_filters_' + str(n_filters) + '_size_' + str(filter_size) + '_lr_' + str(learning_rate) + '.pt'
model_name = 'fromscratch_epochs_' + str(epochs) + '_modes_' + str(modes) + '_pool_' + str(pooling_size) + '_filters_' + str(n_filters) + '_size_' + str(filter_size) + '_lr_' + str(learning_rate) + weighted_loss + '.pt'
model_filepath = os.path.join(CHECKPOINTS_PATH, data_filename, model_name)


## Preprocessing

In [171]:
from antipasti.preprocessing.preprocessing import Preprocessing


In [172]:
df_file = 'abcov_srbd_summary.tsv'

modes = 'all' # Number of normal modes to consider. Relevant if renew_maps is True
renew_maps = False # True to compute again all the normal mode correlation maps
renew_residues = False # True to retrieve again all the chain lengths 
id_category = 'entry'

extra_residues = ['1823', '1177']

In [173]:
if data_name == 'AbCov-SARSCoV2-RBD_original':
    ag_residues = 0
elif data_name == 'AbCov-SARSCoV2-RBD_with-ag':
    ag_residues = 223
elif data_name == 'AbCov-SARSCoV2-RBD_with-offblock':
    ag_residues = 223
elif data_name == 'AbCov-SARSCoV2-RBD_finetune':
    ag_residues = 0
    
    modes = 'all' # Number of normal modes to consider. Relevant if renew_maps is True
    renew_maps = False # True to compute again all the normal mode correlation maps
    renew_residues = False # True to retrieve again all the chain lengths 

    pathological = ['5omm', '5i5k', '1uwx', '1mj7', '1qfw', '1qyg', '4ffz', '3ifl', '3lrh', '3pp4', '3ru8', '3t0w', '3t0x', '4fqr', '4gxu', '4jfx', '4k3h', '4jfz', '4jg0', '4jg1', '4jn2', '4o4y', '4qxt', '4r3s', '4w6y', '4w6y', '5ies', '5ivn', '5j57', '5kvd', '5kzp', '5mes', '5nmv', '5sy8', '5t29', '5t5b', '5vag', '3etb', '3gkz', '3uze', '3uzq', '4f9l', '4gqp', '4r2g', '5c6t', '3fku', '1oau', '1oay']
    scfv = ['4gqp', '3etb', '3gkz', '3uze', '3uzq', '3gm0', '4f9l', '6ejg', '6ejm', '1h8s', '5dfw', '6cbp', '4f9p', '5kov', '1dzb', '5j74', '5aaw', '3uzv', '5aam', '3ux9', '5a2j', '5a2k', '5a2i', '3fku', '5yy4', '3uyp', '5jyl', '1y0l', '1p4b', '3kdm', '4lar', '4ffy', '2ybr', '1mfa', '5xj3', '5xj4', '4kv5', '5vyf'] 
    pathological += scfv

    old_preprocessed_data = Preprocessing(data_path=OLD_DATA_PATH,
                                        chain_lengths_path='chain_lengths/',
                                        dccm_map_path='dccm_maps_full_ags_all/', 
                                        residues_path='lists_of_residues/',
                                        modes=modes, 
                                        pathological=pathological, 
                                        renew_maps=renew_maps, 
                                        renew_residues=renew_residues)


In [ ]:
preprocessed_data = Preprocessing(data_path=DATA_PATH, 
                        structures_path=STRUCTURES_PATH, 
                        df=df_file, 
                        modes=modes, 
                        pathological=extra_residues,
                        chain_lengths_path=CHAIN_LENGTHS_DIR, 
                        dccm_map_path=DCCM_MAPS_DIR, 
                        residues_path = RESIDUES_DIR, 
                        renew_maps=renew_maps, 
                        renew_residues=renew_residues, 
                        alphafold=True, 
                        id_category=id_category, 
                        ag_residues=ag_residues
                        )

In [11]:
if data_name == 'AbCov-SARSCoV2-RBD_with-offblock':
    preprocessed_data.train_x = preprocessed_data.train_x[:, :ag_residues, ag_residues:]
elif data_name == 'AbCov-SARSCoV2-RBD_finetune':
    preprocessed_data.max_res_list_h = old_preprocessed_data.max_res_list_h
    preprocessed_data.max_res_list_l = old_preprocessed_data.max_res_list_l

    preprocessed_data.train_x, preprocessed_data.train_y, preprocessed_data.labels, preprocessed_data.raw_imgs = preprocessed_data.load_training_images()

### Plots

In [12]:
from matplotlib.colors import CenteredNorm

In [160]:
def plot_dccm_maps(preprocessed_data, idx_sample=0):
    
    # Get dimensions
    original_size = preprocessed_data.raw_imgs[idx_sample].shape[0]  # Assuming square matrix
    processed_size = preprocessed_data.train_x.shape[-1]    # Assuming square matrix
    
    # Calculate tick positions
    heavy_len = preprocessed_data.heavy[idx_sample]
    light_len = preprocessed_data.light[idx_sample]
    total_len = heavy_len + light_len
    
    # Left plot ticks
    left_ticks = [0, heavy_len - 1, total_len - 1, original_size - 1]
    left_tick_labels = [1, heavy_len, total_len, original_size]  # Adding +1 to display
    
    # Right plot ticks
    heavy_res_len = len(preprocessed_data.max_res_list_h)
    light_res_len = len(preprocessed_data.max_res_list_l)
    total_res_len = heavy_res_len + light_res_len
    right_ticks = [0, heavy_res_len - 1, total_res_len - 1, processed_size - 1]
    right_tick_labels = [1, heavy_res_len, total_res_len, processed_size]  # Adding +1 to display
    
    # Create figure with width proportional to the matrix sizes
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6),
                                  gridspec_kw={'width_ratios': [original_size, processed_size]})
    
    # Left plot (original)
    im1 = ax1.imshow(preprocessed_data.raw_imgs[idx_sample], origin='lower', cmap='seismic', norm=CenteredNorm())
    ax1.set_xticks(left_ticks)
    ax1.set_yticks(left_ticks)
    ax1.set_xticklabels(left_tick_labels)
    ax1.set_yticklabels(left_tick_labels)
    
    # Right plot (processed)
    im2 = ax2.imshow(preprocessed_data.train_x[idx_sample].reshape(processed_size, processed_size), 
                    origin='lower', cmap='seismic', norm=CenteredNorm())
    ax2.set_xticks(right_ticks)
    ax2.set_yticks(right_ticks)
    ax2.set_xticklabels(right_tick_labels)
    ax2.set_yticklabels(right_tick_labels)
    
    # Colorbar
    # cb2 = fig.colorbar(im2, ax=ax2, fraction=0.045, pad=0.02)
    
    # Adjust tick parameters
    ax1.tick_params(axis='both', which='major', labelsize=10)
    ax2.tick_params(axis='both', which='major', labelsize=10)
    # cb2.ax.tick_params(labelsize=12)
    
    plt.tight_layout()
    plt.show()

In [161]:
selected_entries = np.load(os.path.join(DATA_PATH, CHAIN_LENGTHS_DIR, 'selected_entries.npy'))

In [ ]:
print(preprocessed_data.raw_imgs[0].shape)
print(preprocessed_data.heavy[0])
print(preprocessed_data.light[0])
print(preprocessed_data.train_x[0].shape)

In [ ]:
preprocessed_data[-12]

In [ ]:
selected_entries[-12]

In [ ]:
preprocessed_data.light

In [ ]:
plot_dccm_maps(preprocessed_data)

## Loading test samples

In [229]:
import numpy as np

from matplotlib.colors import CenteredNorm


In [ ]:
train_x, test_x, train_y, test_y, idx_tr, idx_te = torch.load(data_filepath)

input_shape = train_x.shape[-2:]
input_shape

selected_entries = np.load(os.path.join(DATA_PATH, CHAIN_LENGTHS_DIR, 'selected_entries.npy'))

test_entries = selected_entries[idx_te]

test_dccm_map = np.load(os.path.join(DATA_PATH, DCCM_MAPS_DIR, test_entries[-1] + '.npy'))

In [ ]:
input_shape = test_x.shape[-2:]

fig, ax1 = plt.subplots(figsize=(13, 13))
im1 = ax1.imshow(test_x[1].reshape(input_shape[0], input_shape[1]), origin='lower', cmap='seismic', norm=CenteredNorm())

cb1 = plt.colorbar(im1, ax=ax1, fraction=0.045)

ax1.tick_params(axis='both', which='major', labelsize=10)
cb1.ax.tick_params(labelsize=16) 

plt.show()

## Loading training checkpoint

In [238]:
from antipasti.utils.torch_utils import load_checkpoint

In [ ]:
from antipasti.model.model import ANTIPASTI

model = ANTIPASTI(n_filters=n_filters, 
                                filter_size=filter_size, 
                                pooling_size=pooling_size, 
                                input_shape=input_shape, 
                                l1_lambda=2e-3)
model.eval()

In [ ]:
model,optimiser,_,train_losses, test_losses = load_checkpoint(model_filepath, input_shape)
model.eval()

## Predictions

In [250]:
y_pred, outer_layer = model(test_x)

In [ ]:
train_y_pred, inter_filter = model(train_x)

## Results

### Combined Figures

In [109]:
import matplotlib.pyplot as plt
import numpy as np

def plot_combined_figures(y_test, output_test, train_y, train_y_pred, train_losses, test_losses):

    font_size=27
    label_size=30
    tick_size=25

    # Create the combined figure
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 8))
    
    # Plot 1: Test predictions
    ax1.scatter(np.array(output_test), y_test[:,0].detach().numpy())
    corr_test = np.corrcoef(np.array(output_test).T, y_test[:,0].detach().numpy().T)[1,0]
    ax1.plot([-12,-5],[-12,-5], c='r', linestyle='dashed')
    ax1.text(0.05, 0.9, f'R = {corr_test:.4f}', transform=ax1.transAxes, fontsize=label_size, bbox=dict(facecolor='white', alpha=0.8))    
    ax1.set_xlabel('Predicted $log_{10}$($K_d$)', size=font_size)
    ax1.set_ylabel('True $log_{10}$($K_d$)', size=font_size)
    ax1.tick_params(axis='both', which='major', labelsize=tick_size)
    
    # Plot 2: Train predictions
    ax2.scatter(np.array(train_y_pred), train_y[:,0].detach().numpy())
    corr_train = np.corrcoef(np.array(train_y_pred).T, train_y[:,0].detach().numpy().T)[1,0]
    ax2.plot([-12,-5],[-12,-5], c='r', linestyle='dashed')
    ax2.text(0.05, 0.9, f'R = {corr_train:.4f}', transform=ax2.transAxes, fontsize=label_size, bbox=dict(facecolor='white', alpha=0.8))   
    ax2.set_xlabel('Predicted $log_{10}$($K_d$)', size=font_size) 
    ax2.set_ylabel('True $log_{10}$($K_d$)', size=font_size)
    ax2.tick_params(axis='both', which='major', labelsize=tick_size)
    
    # Plot 3: Loss curves
    ax3.plot([test_losses[:][i] for i in range(len(test_losses[:]))])
    ax3.plot([train_losses[:][i] for i in range(len(train_losses[:]))])
    ax3.set_xlabel('Number of epoch', size=font_size)
    ax3.set_ylabel('MSE', size=font_size)
    ax3.legend(['Test', 'Training'], prop={'size': label_size})
    ax3.tick_params(axis='both', which='major', labelsize=tick_size)
    
    plt.tight_layout()
    plt.show()


In [ ]:
plot_combined_figures(test_y, y_pred.detach(), train_y, train_y_pred.detach(), train_losses, test_losses)

### Pred vs true

In [252]:
def plot_true_pred(y_test, output_test, title_size=20, font_size=14):
    fig = plt.figure(figsize=(10, 8))
    plt.scatter(np.array(output_test), y_test[:,0].detach().numpy())
    corr = np.corrcoef(np.array(output_test).T, y_test[:,0].detach().numpy().T)[1,0]
    plt.plot([-12,-5],[-12,-5], c='r', linestyle='dashed')
    plt.title('R = '+str(corr), size=title_size)
    plt.xlabel('Predicted $log_{10}$($K_d$)', size=font_size)
    plt.ylabel('True $log_{10}$($K_d$)', size=font_size)
    plt.show()

In [ ]:
plot_true_pred(test_y, y_pred.detach())

In [ ]:
plot_true_pred(train_y, train_y_pred.detach())

### Loss

In [254]:
# def plot_loss(train_losses, test_losses, title_size=20, font_size=14):
#     fig = plt.figure(figsize=(10, 8))
#     plt.plot([test_losses[:][i] for i in range(len(test_losses[:]))])
#     plt.plot([train_losses[:][i] for i in range(len(train_losses[:]))])
#     plt.xlabel('Number of epoch', size=font_size)
#     plt.ylabel('MSE', size=font_size)
#     plt.legend(['Test', 'Training'], prop={'size': font_size})
#     plt.show()
def plot_loss(train_losses, title_size=20, font_size=14):
    fig = plt.figure(figsize=(10, 8))
    # plt.plot([test_losses[:][i] for i in range(len(test_losses[:]))])
    plt.plot([train_losses[:][i] for i in range(len(train_losses[:]))])
    plt.xlabel('Number of epoch', size=font_size)
    plt.ylabel('Loss', size=font_size)
    # plt.legend(['Test', 'Training'], prop={'size': font_size})
    plt.show()

In [ ]:
plot_loss(train_losses)

### Error vs confidence

In [80]:
import pandas as pd

In [ ]:
df_confidences = pd.read_csv(os.path.join(DATA_PATH, 'confidence/af3_srbd_confidence_summary.csv'), header=0, dtype={'entry': str})

df_confidences['max_pae_abonly'].describe()

test_confidences = df_confidences[df_confidences['entry'].isin(test_entries)]
test_confidences['entry'] = pd.Categorical(test_confidences['entry'], categories=test_entries, ordered=True)
test_confidences = test_confidences.sort_values('entry')

In [ ]:
plt.hist(test_confidences['max_pae_abonly'])

In [381]:
error = abs(y_pred.detach().numpy().reshape(-1) - test_y.numpy().reshape(-1))

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(test_confidences['mean_plddt_abonly'], error, alpha=0.7)
plt.xlabel('Mean pLDDT')
plt.ylabel('Error')
plt.title('Error vs Mean pLDDT')
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(test_confidences['max_pae_abonly'], error, alpha=0.7)
plt.xlabel('Max PAE')
plt.ylabel('Error')
plt.title('Error vs Max PAE')
plt.grid(True)
plt.show()


### Outer layer and importance

In [19]:
import itertools

from antipasti.utils.explaining_utils import compute_umap, compute_region_importance, compute_residue_importance, get_test_contribution, get_maps_of_interest, get_output_representations, plot_map_with_regions


In [20]:
inter_filter_numpy = inter_filter.detach().numpy()

In [ ]:
inter_filter_numpy.shape

In [ ]:
fig, axs = plt.subplots((n_filters+1)//2, 2, figsize=(18, 20))
# size_le = int(np.sqrt(model.fc1.weight.data.numpy().shape[-1] / n_filters))
size_le = ((train_x.size()[2]-model.filter_size-model.pooling_size+2), (train_x.size()[3]-model.filter_size-model.pooling_size+2))
learnt_filter = np.zeros(size_le)

for i, j in itertools.product(range((n_filters+1)//2), range(2)):
    if j == 1 and i == (n_filters+1)//2-1 and n_filters % 2 != 0:
        im_ = learnt_filter
    else:
        im_ = np.multiply(np.mean(inter_filter_numpy, axis=0)[2*i+j], model.fc1.weight.data.numpy().reshape(n_filters,size_le[0]*size_le[1])[2*i+j].reshape(size_le))
        learnt_filter += im_
    im = axs[i,j].imshow(im_, origin='lower', cmap='seismic', norm=CenteredNorm())

In [ ]:
learnt_filter.shape

In [22]:
mean_learnt, mean_image, mean_diff_image = get_maps_of_interest(preprocessed_data, learnt_filter)

In [ ]:
mean_learnt.shape

In [29]:
def plot_map(map):
    input_shape = map.shape[-2:]

    fig, ax1 = plt.subplots(figsize=(13, 13))
    im1 = ax1.imshow(map.reshape(input_shape[0], input_shape[1]), origin='lower', cmap='seismic', norm=CenteredNorm())

    cb1 = plt.colorbar(im1, ax=ax1, fraction=0.045)

    ax1.tick_params(axis='both', which='major', labelsize=10)
    cb1.ax.tick_params(labelsize=16) 

    plt.show()

In [ ]:
# plot_map_with_regions(preprocessed_data, mean_learnt, 'Average of learnt representations')
plot_map(mean_learnt)

In [ ]:
# plot_map_with_regions(preprocessed_data, mean_image, 'Mean normal mode correlation map')

In [ ]:
plot_map_with_regions(preprocessed_data, mean_diff_image, 'Mean difference normal mode correlation map')

In [ ]:
type_of_antigen = 0
compute_region_importance(preprocessed_data, model, type_of_antigen, nanobodies, mode='region')

### Other plots

In [ ]:
selected_entries

In [ ]:
test_confidences = df_confidences[df_confidences['entry'].isin(test_entries)]
test_confidences['entry'] = pd.Categorical(test_confidences['entry'], categories=test_entries, ordered=True)
test_confidences = test_confidences.sort_values('entry')

In [ ]:
confidence_threshold = 31
max_pae_test_confidences = test_confidences['max_pae_abonly'].values
confident_tests = np.where(max_pae_test_confidences < confidence_threshold)[0]

In [ ]:
max_pae_test_confidences

In [ ]:
confident_tests

In [ ]:
plot_true_pred(test_y[confident_tests], y_pred.detach()[confident_tests])

# Model comparisons

In [5]:
import numpy as np

from antipasti.utils.torch_utils import load_checkpoint

import pandas as pd

In [195]:
# data_name = 'AbCov-SARSCoV2-RBD_with-ag'
# data_name = 'AbCov-SARSCoV2-RBD_finetune'
# data_name = 'AbCov-SARSCoV2-RBD_original'

# downweighting = 1
# weighted_loss = f"_downweighting_{downweighting}"

# model_name = 'finetunedmodel_epochs_' + str(epochs) + '_modes_' + str(modes) + '_pool_' + str(pooling_size) + '_filters_' + str(n_filters) + '_size_' + str(filter_size) + '.pt'

In [6]:
def data_inference(data_name, seed=0, confident=None, test_size=0.1):

    if confident:
        data_filename = data_name + '-training-data_seed_' + str(seed) + '_confident_' + str(confident) + '_split_' + str(test_size)
    else:
        data_filename = data_name + '-training-data_seed_' + str(seed) + '_split_' + str(test_size)

    data_filepath = os.path.join(DATA_PATH, TORCH_DIR, data_filename + '.pt')

    train_x, test_x, train_y, test_y, idx_tr, idx_te = torch.load(data_filepath)
    input_shape = train_x.shape[-2:]

    data_dict = {'data_name': data_name,
                'input_shape': input_shape,
                'seed': seed,
                'confident': confident,
                'test_size': test_size}

    return data_filename, train_x, test_x, train_y, test_y, data_dict
    

In [7]:
def model_inference(data_dict,
                    data_filename,
                    train_x,
                    test_x,
                    train_y,
                    test_y,
                    epochs=300, 
                    modes='all', 
                    n_filters=4, 
                    filter_size=4, 
                    pooling_size=1,
                    learning_rate=5e-5,
                    downweighting=0):
    
    if data_dict['data_name'].endswith('finetune'):
        model_name = f'finetunedmodel_epochs_{epochs}_modes_{modes}_pool_{pooling_size}_filters_{n_filters}_size_{filter_size}.pt'
    elif downweighting != 0:
        model_name = f'fromscratch_epochs_{epochs}_modes_{modes}_pool_{pooling_size}_filters_{n_filters}_size_{filter_size}_lr_{learning_rate}_downweighting_{downweighting}.pt'
    else:
        model_name = f'fromscratch_epochs_{epochs}_modes_{modes}_pool_{pooling_size}_filters_{n_filters}_size_{filter_size}_lr_{learning_rate}.pt'

    model_filepath = os.path.join(CHECKPOINTS_PATH, data_filename, model_name)

    try:
        model, optimiser, _, train_losses, test_losses = load_checkpoint(model_filepath, data_dict['input_shape'])
    except Exception as e:
        print(f"Failed to load checkpoint for {model_name}: {e}")
        return None
    
    y_pred, outer_layer = model(test_x)
    y_hat = np.array(y_pred.detach()).T
    y_test = test_y[:,0].detach().numpy().T

    # Calculate correlations
    test_corr = np.corrcoef(y_hat, y_test)[1,0]
    test_mse = np.mean((y_test - y_hat) ** 2)
    
    # Get predictions on training data for train correlation
    with torch.no_grad():
        output_train = model(train_x)
        y_train_hat = np.array(output_train[0].detach()).T
        y_train = train_y[:,0].detach().numpy().T
        train_corr = np.corrcoef(y_train_hat, y_train)[1,0]
        train_mse = np.mean((y_train - y_train_hat) ** 2)
    
    model_dict = {'modes': modes,
            'epochs': epochs,
            'filter_size': filter_size,
            'n_filters': n_filters,
            'pooling_size': pooling_size,
            'learning_rate': learning_rate,
            'downweighting': downweighting,
            'final_train_loss': train_losses[-1],
            'train_corr': train_corr,
            'test_corr': test_corr,
            'train_mse': train_mse,
            'test_mse': test_losses[-1],
            }
    results_dict = data_dict | model_dict

    return results_dict
    

In [8]:
def epochs_checker(data_name):
    if data_name == 'AbCov-SARSCoV2-RBD_original':
        epochs = 100
    elif data_name == 'AbCov-SARSCoV2-RBD_with-ag':
        epochs = 100
    elif data_name == 'AbCov-SARSCoV2-RBD_with-offblock':
        epochs = 300
    elif data_name == 'AbCov-SARSCoV2-RBD_finetune':
        epochs = 500
    return epochs


In [2]:
results = []

In [ ]:
# initial comparison table
hyperparams = {
    'seed': range(5),
    'data_name': ['AbCov-SARSCoV2-RBD_original', 'AbCov-SARSCoV2-RBD_with-ag', 'AbCov-SARSCoV2-RBD_with-offblock', 'AbCov-SARSCoV2-RBD_finetune'],
    'test_size': [0.05, 0.1, 0.2],
    'epochs': [100, 300, 500],
}

modes = 'all'
n_filters = 4
pooling_size = 1
filter_size = 4
learning_rate = 5e-5

# Loading and inference
for data_name in hyperparams['data_name']:
    for test_size in hyperparams['test_size']:
        for seed in hyperparams['seed']:

            data_filename, train_x, test_x, train_y, test_y, data_dict = data_inference(data_name, seed=seed, test_size=test_size)

            epochs = epochs_checker(data_name)
                
            results_dict = model_inference(data_dict,
                                            data_filename,
                                            train_x,
                                            test_x,
                                            train_y,
                                            test_y,
                                            epochs=epochs,
                                            modes=modes,
                                            n_filters=n_filters,
                                            filter_size=filter_size,
                                            pooling_size=pooling_size,
                                            learning_rate=learning_rate)
            if results_dict:
                results.append(results_dict)

In [ ]:
# confident comparisons
hyperparams = {
    'data_name': ['AbCov-SARSCoV2-RBD_original', 'AbCov-SARSCoV2-RBD_with-ag', 'AbCov-SARSCoV2-RBD_finetune'],
    'epochs': [100, 500],
}

seed = 0
test_size = 0.1
confident = 30.7

modes = 'all'
n_filters = 4
pooling_size = 1
filter_size = 4
learning_rate = 5e-5

# Loading and inference
for data_name in hyperparams['data_name']:
    data_filename, train_x, test_x, train_y, test_y, data_dict = data_inference(data_name, seed=seed, test_size=test_size, confident=confident)

    for epochs in hyperparams['epochs']:
        
        results_dict = model_inference(data_dict,
                                        data_filename,
                                        train_x,
                                        test_x,
                                        train_y,
                                        test_y,
                                        epochs=epochs,
                                        modes=modes,
                                        n_filters=n_filters,
                                        filter_size=filter_size,
                                        pooling_size=pooling_size,
                                        learning_rate=learning_rate)
        if results_dict:
            results.append(results_dict)

In [57]:
# hyperparams = {
#     'epochs': [100, 500]
# }

# data_name = 'AbCov-SARSCoV2-RBD_with-offblock'
# seed = 0
# test_size = 0.1

# modes = 'all'
# n_filters =  4
# pooling_size = 1
# filter_size = 4
# learning_rate = 5e-5

# downweighting = 1

# # Loading and inference
# data_filename, train_x, test_x, train_y, test_y, data_dict = data_inference(data_name, seed=seed, test_size=test_size)


# for epochs in hyperparams['epochs']:
#     results_dict = model_inference(data_dict,
#                                 data_filename,
#                                 train_x,
#                                 test_x,
#                                 train_y,
#                                 test_y,
#                                 epochs=epochs,
#                                 modes=modes,
#                                 n_filters=n_filters,
#                                 filter_size=filter_size,
#                                 pooling_size=pooling_size,
#                                 learning_rate=learning_rate,
#                                 downweighting =downweighting)
#     if results_dict:
#         results.append(results_dict)

In [ ]:
# downweighting

hyperparams = {
    'downweighting': [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 6, 8]
}

data_name = 'AbCov-SARSCoV2-RBD_with-offblock'
seed = 0
confident = 30.7
test_size = 0.1

epochs = 300
modes = 'all'
n_filters =  4
pooling_size = 1
filter_size = 4
learning_rate = 5e-5

# Loading and inference
data_filename, train_x, test_x, train_y, test_y, data_dict = data_inference(data_name, seed=seed, confident=confident, test_size=test_size)


for downweighting in hyperparams['downweighting']:
    results_dict = model_inference(data_dict,
                                data_filename,
                                train_x,
                                test_x,
                                train_y,
                                test_y,
                                epochs=epochs,
                                modes=modes,
                                n_filters=n_filters,
                                filter_size=filter_size,
                                pooling_size=pooling_size,
                                learning_rate=learning_rate,
                                downweighting=downweighting)
    if results_dict:
        results.append(results_dict)

In [59]:
# hyperparams = {
#     'epochs': [100, 300, 500],
#     'n_filters': [4, 8, 16],
#     'pooling_size': [1, 2],
#     'learning_rate': [5e-5, 1e-4, 5e-4]
# }

# data_name = 'AbCov-SARSCoV2-RBD_with-offblock'
# seed = 0
# test_size = 0.1

# modes = 'all'
# filter_size = 4

# # Loading and inference
# data_filename, train_x, test_x, train_y, test_y, data_dict = data_inference(data_name, seed=seed, test_size=test_size)

# for epochs in hyperparams['epochs']:
#     for n_filters in hyperparams['n_filters']:
#         for pooling_size in hyperparams['pooling_size']:
#             for learning_rate in hyperparams['learning_rate']:
#                 results_dict = model_inference(data_dict,
#                                 data_filename,
#                                 train_x,
#                                 test_x,
#                                 train_y,
#                                 test_y,
#                                 epochs=epochs,
#                                 modes=modes,
#                                 n_filters=n_filters,
#                                 filter_size=filter_size,
#                                 pooling_size=pooling_size,
#                                 learning_rate=learning_rate)
#                 if results_dict:
#                     results.append(results_dict)

In [ ]:
# hyperparams model selection
hyperparams = {
    'data_name': ['AbCov-SARSCoV2-RBD_with-offblock', 'AbCov-SARSCoV2-RBD_with-ag'],
    'downweighting': [0, 2],
    'n_filters': [2, 4, 8],
    'pooling_size': [1, 2],
    'learning_rate': [5e-5, 1e-4, 5e-4]
}

seed = 0
confident = 30.7
test_size = 0.1

modes = 'all'
filter_size = 4

for data_name in hyperparams['data_name']:
    # Loading and inference
    data_filename, train_x, test_x, train_y, test_y, data_dict = data_inference(data_name, seed=seed, test_size=test_size, confident=confident)

    for downweighting in hyperparams['downweighting']:
        for n_filters in hyperparams['n_filters']:
            for pooling_size in hyperparams['pooling_size']:
                for learning_rate in hyperparams['learning_rate']:
                    if data_name == 'AbCov-SARSCoV2-RBD_with-offblock':
                        epochs = 300
                    else:
                        epochs = 100
                    results_dict = model_inference(data_dict,
                                    data_filename,
                                    train_x,
                                    test_x,
                                    train_y,
                                    test_y,
                                    epochs=epochs,
                                    modes=modes,
                                    n_filters=n_filters,
                                    filter_size=filter_size,
                                    pooling_size=pooling_size,
                                    learning_rate=learning_rate,
                                    downweighting=downweighting)
                    if results_dict:
                        results.append(results_dict)

In [ ]:
results_df = pd.DataFrame(results)

print(results_df)

In [14]:
results_df.to_csv('./results.csv', index=False)

## Analysis

### Initial averaging

In [11]:
initial_df = results_df[['data_name', 'epochs', 'test_size', 'seed', 'train_corr', 'test_corr', 'train_mse', 'test_mse']]

In [12]:
initial_df = initial_df.groupby(['data_name','epochs', 'test_size']).agg({
    'train_corr': 'mean', 
    'test_corr': 'mean', 
    'train_mse': 'mean', 
    'test_mse': 'mean'}).reset_index()

In [13]:
initial_df.to_csv('./initial_results.csv', index=False)

### Downweighting

In [146]:
import matplotlib.pyplot as plt

In [ ]:
# Downweighting performance plots

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), dpi=300)

# Correlation plot
ax1.plot(results_df['downweighting'], results_df['train_corr'], 
        'o-', label='Train', markersize=5, linewidth=1.5)
ax1.plot(results_df['downweighting'], results_df['test_corr'], 
        's--', label='Test', markersize=5, linewidth=1.5)
ax1.set(xlabel=r'$\beta$', ylabel='R', xticks=np.arange(0, 8.5, 1))
ax1.legend(frameon=True, loc='best')

# MSE plot
ax2.plot(results_df['downweighting'], results_df['train_mse'], 
         'o-', label='Train', markersize=5, linewidth=1.5)
ax2.plot(results_df['downweighting'], results_df['test_mse'], 
         's--', label='Test', markersize=5, linewidth=1.5)
ax2.set(xlabel=r'$\beta$', ylabel='MSE', xticks=np.arange(0, 8.5, 1))
ax2.legend(frameon=True, loc='best')

# Adjust layout
plt.tight_layout()
plt.show()